# Avaliação: Qwen3-1.7B base × `Er1ckRg/cuphead_basico`

Compara o modelo base com o modelo ajustado nos **mesmos prompts e com os mesmos parâmetros de geração**:

- **Prompts:** perguntas de `eval/test_prompts.jsonl`, escritas para este teste. Uma checagem automática confirma que não se sobrepõem ao dataset de treino.
- **Decodificação:** gulosa e determinística (`do_sample=False`), com o mesmo `max_new_tokens`, o mesmo dtype e o mesmo system prompt do treino.
- **Raciocínio:** `enable_thinking=False` no chat template dos dois modelos. O treino (TRL 1.7.0, template `qwen3_training.jinja`) incluiu um bloco `<think></think>` vazio dentro da loss. O modelo ajustado aprendeu a não raciocinar, e esse é o formato que `enable_thinking=False` produz.

**Como rodar no Kaggle:** *Settings → Accelerator: GPU (T4 ou P100)*, *Internet: On*. Depois *Run All*. Os modelos são públicos, então não é preciso token.
As saídas ficam em `/kaggle/working/`:
- `eval_responses.csv`: respostas lado a lado.
- `eval_scores.csv`: nota por prompt e por modelo, com os fatos que faltaram.
- `eval_summary.md`: tabelas prontas para o model card.
- `eval_run_config.json`: configuração da execução.

In [ ]:
import os

# No Kaggle, alinha as versões com as do treino. Defina SKIP_PIP=1 para pular (ex.: execução local).
if not os.environ.get("SKIP_PIP"):
    %pip install -q "transformers==5.12.1" "accelerate==1.14.0" "huggingface_hub==1.21.0"

In [ ]:
import os
from pathlib import Path

# Parâmetros (variáveis de ambiente permitem um teste rápido com modelos minúsculos)
BASE_MODEL_ID = os.environ.get("BASE_MODEL_ID", "Qwen/Qwen3-1.7B")
FINETUNED_MODEL_ID = os.environ.get("FINETUNED_MODEL_ID", "Er1ckRg/cuphead_basico")

DATASET_REPO_ID = "Er1ckRg/cuphead_basico"
DATASET_FILENAME = "dataset.jsonl"
DATASET_REVISION = "dd714567cfd0ff14195bd04d2b05b7192f5d1b84"

PROMPTS_URL = "https://raw.githubusercontent.com/ErickRibeiroG/CupLLM/main/eval/test_prompts.jsonl"

# Mesmo system prompt de todos os exemplos de treino. É usado nos DOIS modelos.
SYSTEM_PROMPT = (
    "Você é um especialista no jogo Cuphead e sua DLC The Delicious Last Course. "
    "Responda perguntas de forma precisa e factual sobre a campanha, chefes, mecânicas e conteúdo do jogo."
)

ENABLE_THINKING = False
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", 512))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", 8))

OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "/kaggle/working" if Path("/kaggle/working").exists() else "eval/results"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Conjunto de teste

In [ ]:
import json
import urllib.request


def find_prompts_file():
    candidates = [Path("eval/test_prompts.jsonl"), Path("test_prompts.jsonl")]
    if Path("/kaggle/input").exists():
        candidates += sorted(Path("/kaggle/input").rglob("test_prompts.jsonl"))
    return next((p for p in candidates if p.exists()), None)


prompts_path = find_prompts_file()
if prompts_path is not None:
    lines = prompts_path.read_text(encoding="utf-8").splitlines()
    print(f"Prompts carregados de {prompts_path}")
else:
    lines = urllib.request.urlopen(PROMPTS_URL).read().decode("utf-8").splitlines()
    print(f"Prompts carregados de {PROMPTS_URL}")

test_set = [json.loads(line) for line in lines if line.strip()]
assert len({item["id"] for item in test_set}) == len(test_set), "ids duplicados"
assert 20 <= len(test_set) <= 50, len(test_set)

from collections import Counter
print(len(test_set), "prompts:", dict(Counter(item["category"] for item in test_set)))

## 2. Checagem de sobreposição com o treino

Cada pergunta de teste é comparada com **todas** as 84 perguntas do dataset (treino e validação), depois de normalizar caixa, acentos e pontuação. A checagem usa três critérios:
- **igualdade exata** depois da normalização;
- **similaridade de caracteres** (`difflib.SequenceMatcher`) ≥ 0,80;
- **Jaccard de palavras de conteúdo** (sem stopwords) ≥ 0,60.

Qualquer par que passe de um limite interrompe o notebook.

In [ ]:
import difflib
import re
import unicodedata

from huggingface_hub import hf_hub_download

def normalize(text):
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", text)).strip()


# "cuphead" e "jogo" aparecem em quase todas as perguntas e não distinguem o assunto.
STOPWORDS = set(normalize(
    "a o as os um uma uns umas de do da dos das em no na nos nas e é ou que qual quais quem como "
    "por para com se sua seu suas seus ao aos à às existe existem tem possui ser são foi há cuphead jogo"
).split())


def content_tokens(text):
    return {tok for tok in normalize(text).split() if tok not in STOPWORDS}


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0


dataset_path = hf_hub_download(DATASET_REPO_ID, DATASET_FILENAME, repo_type="dataset", revision=DATASET_REVISION)
train_rows = [json.loads(line) for line in open(dataset_path, encoding="utf-8") if line.strip()]
train_questions = [next(m["content"] for m in row["messages"] if m["role"] == "user") for row in train_rows]
print("Perguntas no dataset de treino:", len(train_questions))

train_norm = [(tq, normalize(tq), content_tokens(tq)) for tq in train_questions]
overlap_rows = []
for item in test_set:
    q_norm, q_tok = normalize(item["question"]), content_tokens(item["question"])
    scores = [(difflib.SequenceMatcher(None, q_norm, tn).ratio(), jaccard(q_tok, tt), tq) for tq, tn, tt in train_norm]
    best_ratio, _, closest = max(scores)
    overlap_rows.append({
        "id": item["id"], "question": item["question"], "closest_train_question": closest,
        "char_similarity": round(best_ratio, 3), "max_jaccard": round(max(s[1] for s in scores), 3),
        "exact_match": any(q_norm == tn for _, tn, _ in train_norm),
    })

import pandas as pd

overlap_df = pd.DataFrame(overlap_rows).sort_values("char_similarity", ascending=False)
print(overlap_df.head(8).to_string())

violations = overlap_df[(overlap_df.exact_match) | (overlap_df.char_similarity >= 0.80) | (overlap_df.max_jaccard >= 0.60)]
assert violations.empty, f"Sobreposição com o treino:\n{violations.to_string()}"
print(f"OK: nenhuma sobreposição (max similaridade = {overlap_df.char_similarity.max():.3f}, "
      f"max Jaccard = {overlap_df.max_jaccard.max():.3f})")
# Pares próximos do limite costumam compartilhar só o molde da pergunta ("Como funciona a arma X?"),
# não o assunto. Confira a tabela acima.

## 3. Geração com parâmetros idênticos nos dois modelos

In [ ]:
import gc
import time

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
# bf16 só com suporte nativo (Ampere+). Em T4/P100, fp32 evita overflow do fp16. Mesmo dtype nos dois modelos.
DTYPE = torch.bfloat16 if device == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float32
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "", "| dtype:", DTYPE)


def build_prompt(tokenizer, question):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING
    )


@torch.no_grad()
def generate_answers(model_id, questions):
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=DTYPE).to(device).eval()

    # GenerationConfig explícito: ignora temperature/top_p/top_k do generation_config.json de cada repo.
    gen_config = GenerationConfig(
        do_sample=False,
        max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=model.generation_config.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    prompts = [build_prompt(tokenizer, q) for q in questions]
    answers, n_tokens = [], []
    start = time.time()
    for i in range(0, len(prompts), BATCH_SIZE):
        batch = tokenizer(prompts[i : i + BATCH_SIZE], return_tensors="pt", padding=True, add_special_tokens=False).to(device)
        output = model.generate(**batch, generation_config=gen_config)
        for row in output[:, batch["input_ids"].shape[1] :]:
            new_tokens = row[row != tokenizer.pad_token_id]
            n_tokens.append(int(new_tokens.shape[0]))
            answers.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    elapsed = time.time() - start

    run_info = {"model_id": model_id, "seconds": round(elapsed, 1),
                "generation_config": gen_config.to_diff_dict(), "prompt_example": prompts[0]}
    del model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    return answers, n_tokens, run_info


questions = [item["question"] for item in test_set]
base_answers, base_tokens, base_info = generate_answers(BASE_MODEL_ID, questions)
ft_answers, ft_tokens, ft_info = generate_answers(FINETUNED_MODEL_ID, questions)
assert base_info["generation_config"] == ft_info["generation_config"], "parâmetros de geração diferentes"
assert base_info["prompt_example"] == ft_info["prompt_example"], "prompts formatados diferentes"
print(f"base: {base_info['seconds']}s | ajustado: {ft_info['seconds']}s")
print("Prompt formatado (exemplo):\n" + base_info["prompt_example"])

## 4. Respostas lado a lado

In [ ]:
results = pd.DataFrame(
    {
        "id": [item["id"] for item in test_set],
        "category": [item["category"] for item in test_set],
        "question": questions,
        "base_answer": base_answers,
        "finetuned_answer": ft_answers,
        "base_new_tokens": base_tokens,
        "finetuned_new_tokens": ft_tokens,
    }
)
results["base_hit_max_tokens"] = results.base_new_tokens >= MAX_NEW_TOKENS
results["finetuned_hit_max_tokens"] = results.finetuned_new_tokens >= MAX_NEW_TOKENS
results["base_has_think"] = results.base_answer.str.contains("<think>", regex=False)
results["finetuned_has_think"] = results.finetuned_answer.str.contains("<think>", regex=False)

csv_path = OUTPUT_DIR / "eval_responses.csv"
results.to_csv(csv_path, index=False, encoding="utf-8")

run_config = {
    "base_model": BASE_MODEL_ID,
    "finetuned_model": FINETUNED_MODEL_ID,
    "dataset": f"{DATASET_REPO_ID}@{DATASET_REVISION}",
    "n_prompts": len(test_set),
    "system_prompt": SYSTEM_PROMPT,
    "enable_thinking": ENABLE_THINKING,
    "dtype": str(DTYPE),
    "device": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
    "batch_size": BATCH_SIZE,
    "generation_config": base_info["generation_config"],
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
    "seconds": {"base": base_info["seconds"], "finetuned": ft_info["seconds"]},
}
(OUTPUT_DIR / "eval_run_config.json").write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

print("Salvo em", csv_path)
print(results[["base_new_tokens", "finetuned_new_tokens"]].describe().round(1).to_string())
print("Respostas cortadas por max_new_tokens:", int(results.base_hit_max_tokens.sum()), "(base) /",
      int(results.finetuned_hit_max_tokens.sum()), "(ajustado)")

In [ ]:
for _, row in results.head(5).iterrows():
    print(f"### [{row.id}] {row.question}\n--- BASE ---\n{row.base_answer}\n--- AJUSTADO ---\n{row.finetuned_answer}\n")

## 5. Métrica: cobertura de fatos-chave

Cada prompt tem de 1 a 4 **fatos-chave** em `key_facts`. Cada fato lista variantes aceitas separadas por `|`. Um `*` no fim da variante aceita prefixo (`abelha*` casa "abelhas"). A comparação é feita no texto normalizado (sem caixa, acentos ou pontuação), com limite de palavra.

- **Nota da resposta** = fração dos fatos-chave presentes. **Completa** = todos presentes.
- **Por quê:** a tarefa é QA factual sobre entidades (nomes de chefes, itens, plataformas). A métrica é determinística, barata e auditável. ROUGE/BLEU premiariam imitar o *estilo* do dataset, não acertar o fato.
- **Limitações:** não pune informação extra ou errada (alucinação) e é frágil a sinônimos fora da lista. Por isso o tamanho das respostas é reportado junto, e as respostas devem ser lidas no CSV.
- **Cobertura no treino:** cada prompt também é marcado como `coberto`, `parcial` ou `novo`, conforme os seus fatos-chave apareçam ou não nos exemplos de treino sobre o mesmo assunto (campo `subject`). Isso separa "aprendeu fatos do dataset" de "generalizou".

In [ ]:
# Permite recalcular a métrica a partir do CSV salvo, sem gerar as respostas de novo.
if "results" not in globals():
    results = pd.read_csv(OUTPUT_DIR / "eval_responses.csv", keep_default_na=False)


def compile_fact(fact):
    patterns = []
    for variant in fact.split("|"):
        variant = variant.strip()
        is_prefix = variant.endswith("*")
        core = normalize(variant.rstrip("*"))
        patterns.append(r"\b" + re.escape(core) + ("" if is_prefix else r"\b"))
    return re.compile("|".join(patterns))


def fact_hits(answer, facts):
    text = normalize(answer)
    return [bool(compile_fact(f).search(text)) for f in facts]


facts_by_id = {item["id"]: item["key_facts"] for item in test_set}
unvalidated = [item["id"] for item in test_set if not item.get("validated")]

# Cobertura no treino: um fato só conta como "coberto" se aparecer na resposta de um exemplo
# de treino que trata do mesmo assunto (campo `subject`, buscado na pergunta + resposta).
# Sem essa restrição, "sereia" na resposta sobre Brineybeard marcaria Cala Maria como coberta.
train_pairs_norm = [
    (normalize(" ".join(m["content"] for m in row["messages"] if m["role"] != "system")),
     normalize(next(m["content"] for m in row["messages"] if m["role"] == "assistant")))
    for row in train_rows
]


def training_coverage(item):
    if item["category"] == "fora_do_dominio":
        return "fora_do_dominio"
    subject = compile_fact(item["subject"])
    answers = [answer for pair, answer in train_pairs_norm if subject.search(pair)]
    covered = [any(compile_fact(f).search(a) for a in answers) for f in item["key_facts"]]
    return "coberto" if all(covered) else ("novo" if not any(covered) else "parcial")


coverage = {item["id"]: training_coverage(item) for item in test_set}

rows = []
for _, r in results.iterrows():
    facts = facts_by_id[r["id"]]
    for model_name, col in [("base", "base_answer"), ("finetuned", "finetuned_answer")]:
        hits = fact_hits(r[col], facts)
        rows.append({
            "id": r["id"], "category": r["category"], "training_coverage": coverage[r["id"]], "model": model_name,
            "score": sum(hits) / len(hits), "complete": all(hits),
            "missing_facts": " ; ".join(f for f, h in zip(facts, hits) if not h),
            "new_tokens": r[f"{model_name}_new_tokens"],
        })
scores = pd.DataFrame(rows)
scores.to_csv(OUTPUT_DIR / "eval_scores.csv", index=False, encoding="utf-8")

In [ ]:
def summarize(df, by=None):
    agg = dict(n=("id", "nunique"), fact_coverage=("score", "mean"), complete=("complete", "mean"), mean_tokens=("new_tokens", "mean"))
    keys = ([by] if by else []) + ["model"]
    out = df.groupby(keys).agg(**agg).reset_index()
    out["fact_coverage"] = (out.fact_coverage * 100).round(1)
    out["complete"] = (out.complete * 100).round(1)
    out["mean_tokens"] = out.mean_tokens.round(0).astype(int)
    return out


in_domain = scores[scores.category != "fora_do_dominio"]
overall = summarize(in_domain).assign(subset="Cuphead (domínio)")
general = summarize(scores[scores.category == "fora_do_dominio"]).assign(subset="Fora do domínio")
summary = pd.concat([overall, general])[["subset", "model", "n", "fact_coverage", "complete", "mean_tokens"]]
by_category = summarize(scores, "category")
by_coverage = summarize(in_domain, "training_coverage")

# Comparação pareada (mesmo prompt): quantas vezes o ajustado ganhou/empatou/perdeu
paired = in_domain.pivot(index="id", columns="model", values="score")
wins, ties, losses = int((paired.finetuned > paired.base).sum()), int((paired.finetuned == paired.base).sum()), int((paired.finetuned < paired.base).sum())

for title, df in [("Resumo", summary), ("Por categoria", by_category), ("Por cobertura no treino", by_coverage)]:
    print(f"\n=== {title} ===\n" + df.to_string(index=False))
print(f"\nPareado (domínio, n={len(paired)}): ajustado melhor em {wins}, empate em {ties}, pior em {losses}")
print("Prompts por cobertura no treino:", dict(Counter(coverage.values())))
if unvalidated:
    print(f"\nATENÇÃO: {len(unvalidated)} prompts com gabarito ainda não validado (validated=false).")

In [ ]:
# Tabela em Markdown pronta para colar no model card / README
def to_markdown(df):
    header = "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n"
    return header + "\n".join("| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False))


markdown = "\n\n".join([
    "### Summary\n\n" + to_markdown(summary.rename(columns={"fact_coverage": "fact coverage (%)", "complete": "fully correct (%)", "mean_tokens": "mean tokens"})),
    "### By training coverage (in-domain)\n\n" + to_markdown(by_coverage),
    f"Paired comparison (in-domain, n={len(paired)}): fine-tuned better on {wins}, tie on {ties}, worse on {losses}.",
])
(OUTPUT_DIR / "eval_summary.md").write_text(markdown, encoding="utf-8")
print(markdown)
print("\nArquivos:", [p.name for p in OUTPUT_DIR.iterdir() if p.name.startswith("eval_")])